Google Colab setup

In [ ]:
%cd geneforge

[Errno 2] No such file or directory: 'geneforge'
/content


In [1]:
!git clone https://github.com/jordanlgraves/geneforge
%cd geneforge
!git checkout jg

Cloning into 'geneforge'...
remote: Enumerating objects: 1303, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 1303 (delta 67), reused 108 (delta 44), pack-reused 1157 (from 1)
Receiving objects: 100% (1303/1303), 84.03 MiB | 29.88 MiB/s, done.
Resolving deltas: 100% (802/802), done.
/content/geneforge
Branch 'jg' set up to track remote branch 'jg' from 'origin'.
Switched to a new branch 'jg'


In [ ]:
!uv pip install openpipe-art==0.3.11.post2 "gql<4" --prerelease allow --no-cache-dir
!uv pip install langchain_community
!uv pip install -r requirements.txt
!git submodule update --init --recursive

In [1]:
#@title Read/setup env
# %cd ..
%load_ext autoreload
%autoreload 2

# Configure logger to ignore everything to avoid cluttering the output
import logging
logging.getLogger().setLevel(logging.WARNING)

import dotenv # load env vars from .env
dotenv.load_dotenv()

from openai import OpenAI
import dotenv
import os

dotenv.load_dotenv()

False

In [ ]:
import art
import random
from dotenv import load_dotenv
from art.local import LocalBackend
load_dotenv()
random.seed(42)
import os

backend = LocalBackend(in_process=True)

model = art.TrainableModel(
    name="geneforge",
    project="max-promoter-strength",
    base_model="Qwen/Qwen2.5-3B-Instruct",
)
# allenai/Llama-3.1-Tulu-3.1-8B
# Qwen/Qwen3-8B
# meta-llama/Llama-4-Scout-17B-16E-Instruct
# # To run on a T4, we need to override some config defaults.
# model._internal_config = art.dev.InternalModelConfig(
#     init_args=art.dev.InitArgs(
#         max_seq_length=16384, #8192,
#     ),
#     engine_args=art.dev.EngineArgs(
#         enforce_eager=True,
#         gpu_memory_utilization=0.8,
#     ),
# )

await model.register(backend)

INFO 08-01 17:39:17 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 08-01 17:39:18 [__init__.py:239] Automatically detected platform cuda.


/usr/local/lib/python3.11/dist-packages/art/local/state.py:5: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth  # type: ignore


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.1: Fast Qwen2 patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 78.1%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 32768. Num Sequences = 320.
Unsloth: vLLM's KV Cache can use up to 28.67 GB. Also swap space = 6 GB.
INFO 08-01 17:39:43 [config.py:717]

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 08-01 17:39:47 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 08-01 17:39:48 [model_runner.py:1140] Model loading took 2.2564 GiB and 2.858439 seconds
INFO 08-01 17:39:52 [worker.py:287] Memory profiling takes 3.92 seconds
INFO 08-01 17:39:52 [worker.py:287] the current vLLM instance can use total_gpu_memory (39.56GiB) x gpu_memory_utilization (0.78) = 30.89GiB
INFO 08-01 17:39:52 [worker.py:287] model weights take 2.26GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 2.86GiB; the rest of the memory reserved for KV Cache is 25.69GiB.
INFO 08-01 17:39:53 [executor_base.py:112] # cuda blocks: 46764, # CPU blocks: 10922
INFO 08-01 17:39:53 [executor_base.py:117] Maximum concurrency for 32768 tokens per request: 22.83x
INFO 08-01 17:39:57 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.

Capturing CUDA graph shapes:   0%|          | 0/43 [00:00<?, ?it/s]

In [ ]:
#@title Get promoter sequences
import random
from src.scenarios.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

from src.library.cello_library import CelloLibrary
library = CelloLibrary()
library.select_library("Eco1C1G1T1")
promoters = {collection["name"]: collection["dnasequence"] for collection in library.user_constraints if collection["collection"] == "parts" and collection["type"] == "promoter"}
promoter_names = list(promoters.keys())
random.shuffle(promoter_names)
promoter_sequences = list(promoters.keys())

In [ ]:
#@title Run example workflow run

promoter_sequence = promoters[promoter_names[0]]
print(promoter_sequence)
workflow = MaximizePromoterStrengthWorkflow(example_name="maximize_promoter_strength_workflow_test",
                                            promoter_sequence=promoter_sequence,
                                            use_reasoning_model=True,
                                            art_model=model,
                                            llm_client_type='art')
await workflow.run_async()
print(workflow.get_metrics())

In [ ]:
from sklearn.model_selection import train_test_split
from src.scenarios.agent.egc_1p1 import EGCProblem1p1Workflow
from src.scenarios.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

message_lists = []
metrics = []
scenarios = []

train_sequences, eval_sequences = train_test_split(list(promoters.keys()), test_size=0.2, random_state=42)

print('Eval sequences: ', len(eval_sequences))
print('Train sequences: ', len(train_sequences))

In [ ]:
from src.adapters.art_adapter import ArtAdapter
import art
from art.gather import gather_trajectory_groups
from src.scenarios.agent.egc_1p1 import EGCProblem1p1Workflow
from src.scenarios.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow
from art.rewards import ruler
from art import TrainConfig
from art.rewards import ruler_score_group
from src.scenarios.agent.maximize_promoter_strength import GRADING_RUBRIC as RUBRIC_MAXIMIZE_PROMOTER_STRENGTH

training_config = {
    "groups_per_step": 1,
    "num_epochs": 20,
    "rollouts_per_group": 3,
    "learning_rate": 1e-5,
    "max_steps": 20,
    "max_rounds": 3
}

for step in range(training_config["max_steps"]):
  groups = tuple(
      art.TrajectoryGroup(
          (
              ArtAdapter(
                  MaximizePromoterStrengthWorkflow(
                      example_name=f"maximize_promoter_{step}_{promoter_sequence}",
                      promoter_sequence=promoter_sequence,
                      use_reasoning_model=True,
                      art_model=model,
                      llm_client_type='art'
                  ),
                  step=step,
              ).rollout(max_rounds=training_config["max_rounds"])
              for _ in range(training_config["rollouts_per_group"])
          )
      )
      for promoter_sequence in promoter_sequences
  )

  # print metrics
  for group in groups:
    for adapter in group:
      print(f"Metrics: {adapter.workflow.get_metrics()}")

  # run them
  max_promoter_strength_groups = await gather_trajectory_groups(
      groups,
      pbar_desc="gather",
      max_exceptions=18,
  )

  judged_groups = []
  for group in max_promoter_strength_groups:
      rubric = RUBRIC_MAXIMIZE_PROMOTER_STRENGTH
      judged_group = await ruler_score_group(group, "openai/gpt-4o-mini-2024-07-18", debug=True)
      judged_groups.append(judged_group)


  await model.delete_checkpoints()
  await model.train(
      judged_groups,
      config=art.TrainConfig(learning_rate=training_config["learning_rate"]),
      # Lowering the logprob_calculation_chunk_size is a memory saving measure
      # to allow longer sequences (up to 8192 tokens) to be processed on a T4.
      _config={"logprob_calculation_chunk_size": 4},
  )

  print(f'Completed step {step}')

gather:   0%|          | 0/3 [00:00<?, ?it/s]

[RULER] Pretty-printed LLM choice JSON:

{
    'scores': [
        {
            'trajectory_id': '1',
            'explanation': 'This trajectory started to estimate the promoter strength but failed to select a 
library first, leading to an unsuccessful function call. Overall progress was minimal due to early errors.',
            'score': 0.2
        },
        {
            'trajectory_id': '2',
            'explanation': 'This trajectory correctly identified the error regarding library selection and 
proceeded to select the library before estimating the promoter strength. The function calls were clear and logical,
leading to a successful endpoint in estimating the promoter strength.',
            'score': 0.6
        },
        {
            'trajectory_id': '3',
            'explanation': "This trajectory followed a structured approach to assess and optimize the promoter 
sequence. It effectively selected the library and estimated the sequence's strength, achieving the goal efficiently
without detours or errors. It presents a comprehensive method for promoter optimization.",
            'score': 0.8
        }
    ]
}

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: jlgraves (jlgraves-geneforge) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb run initialized! You can view it at https://wandb.ai/jlgraves-geneforge/max-promoter-strength/runs/geneforge


...

{
    'scores': [
        {
            'trajectory_id': '1',
            'explanation': 'Trajectory 1 correctly follows the steps to maximize promoter strength by selecting the
library first before estimating the strength of the original promoter. Though it encounters an error initially, it 
corrects the process and completes actions effectively without unnecessary detours. This trajectory makes efficient
and direct progress towards the goal.',
            'score': 0.9
        },
        {
            'trajectory_id': '2',
            'explanation': 'Trajectory 2 begins with the intent to estimate the promoter strength but fails to 
properly format the JSON and does not handle the library selection first, resulting in multiple errors without real
progress. The trajectory shows some awareness of the problem but lacks efficiency and does not follow through 
effectively, resulting in a lower score.',
            'score': 0.5
        },
        {
            'trajectory_id': '3',
            'explanation': 'Trajectory 3 demonstrates an understanding of the necessary steps, but it continues to 
encounter issues with library selection before estimating the promoter strength, leading to repeated errors. The 
initial steps are sound, but the inability to effectively proceed with the task and focus on the core goal hinders 
overall efficiency and progress.',
            'score': 0.4
        }
    ]
}

Deleted checkpoint ./.art/max-promoter-strength/models/geneforge/0004
Packed 3 trajectories into 1 sequences of length 12288


train:   0%|          | 0/1 [00:00<?, ?it/s]

Completed step 5


gather:   0%|          | 0/3 [00:00<?, ?it/s]